# Exercise 10 : Evaluating a System of Agents

- A final-answer check scores facts in the reply ; it can miss a false claim or reject a paraphrase
- Trajectory match compares tool calls ; strict and unordered catch different failures
- A specialist that passes does not guarantee the desk keeps every fact

You will score a small Aurelia multi-agent desk on six labelled cases : final
answers, trajectory match, and one score per agent. All data is made up.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## Step 1 : The desk and the cases

In [2]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import tool
from langgraph.errors import GraphRecursionError
from rank_bm25 import BM25Okapi

DB = Path(data("capstone/service.db")).resolve()
MANUALS = Path(data_folder("capstone/manuals"))
PAGES = [(p.name, p.read_text(encoding="utf-8")) for p in sorted(Path(MANUALS).glob("*.md"))]
BM25 = BM25Okapi([text.lower().split() for _, text in PAGES])
LIMIT = {"recursion_limit": 24}


@tool
def run_sql(query: str) -> str:
    """Run one read-only SELECT on the Aurelia service database."""
    if not query.lstrip().lower().startswith("select"):
        return "Only SELECT statements are allowed."
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        try:
            return json.dumps(db.execute(query).fetchmany(30))
        except sqlite3.Error as exc:
            return f"SQL error : {exc}"


@tool
def search_manuals(query: str) -> str:
    """Keyword search over Aurelia manuals and bulletins. Returns the three best documents."""
    scores = BM25.get_scores(query.lower().split())
    best = sorted(range(len(PAGES)), key=lambda i: -scores[i])[:3]
    return "\n\n".join(f"[{PAGES[i][0]}]\n{PAGES[i][1][:900]}" for i in best)


SYSTEM = ("You support Aurelia field engineers. Use tools for facts. Be brief. "
          "Preserve part numbers, stock counts and depot names.")
parts_agent = create_agent(llm, tools=[run_sql], system_prompt=SYSTEM, name="parts")
manuals_agent = create_agent(llm, tools=[search_manuals], system_prompt=SYSTEM, name="manuals")
desk = create_agent(llm, tools=[run_sql, search_manuals], system_prompt=SYSTEM, name="desk")

CASES = [json.loads(line) for line in
         Path(data("capstone/eval.jsonl")).read_text(encoding="utf-8").splitlines()]
SELECTED_IDS = ["db-01", "db-03", "man-01", "man-07", "mix-01", "mix-02"]
SELECTED = [c for c in CASES if c["id"] in SELECTED_IDS]

# eval.jsonl's answer_contains mixes alternatives (db-01 : "0" or "not in stock") and
# facts that must all appear (mix-01 : "CT-FN-02" and "6"). REQUIRED makes it explicit :
# every pattern must match ; spellings of one fact share a pattern, joined by | .
# \b keeps "6" from matching "16" or "E-CT-610". A no-stock 0 must sit next to a stock
# word, so the 0 in "firmware 5.0" or "0 open orders" does not count.
NO_STOCK = (r"\b(?:0|zero|no)\s+(?:units?\b[^.]{0,30}?)?(?:in stock|available|stock)\b"
            r"|\bstock(?: level| count| is)?\s*[:=]?\s*(?:0|zero|none)\b"
            r"|not in stock|out of stock|not available|unavailable")
REQUIRED = {
    "db-01": [NO_STOCK],
    "db-03": [r"\b5\.1\b"],
    "man-01": [r"arcing"],
    "man-07": [r"fan filter"],
    "mix-01": [r"CT-FN-02", r"\b6\b|\bsix\b"],
    "mix-02": [r"UX-BT-02", NO_STOCK, r"Zwolle"],
}
print({a.name: [t.name for t in tools] for a, tools in
       ((parts_agent, [run_sql]), (manuals_agent, [search_manuals]), (desk, [run_sql, search_manuals]))})
print(len(SELECTED), "selected cases :", [c["id"] for c in SELECTED])
for c in SELECTED:
    print(f"{c['id']:8} answer_contains {c['answer_contains']}  REQUIRED {REQUIRED[c['id']]}")

{'parts': ['run_sql'], 'manuals': ['search_manuals'], 'desk': ['run_sql', 'search_manuals']}
6 selected cases : ['db-01', 'db-03', 'man-01', 'man-07', 'mix-01', 'mix-02']
db-01    answer_contains ['0', 'not in stock']  REQUIRED ['\\b(?:0|zero|no)\\s+(?:units?\\b[^.]{0,30}?)?(?:in stock|available|stock)\\b|\\bstock(?: level| count| is)?\\s*[:=]?\\s*(?:0|zero|none)\\b|not in stock|out of stock|not available|unavailable']
db-03    answer_contains ['5.1']  REQUIRED ['\\b5\\.1\\b']
man-01   answer_contains ['arcing']  REQUIRED ['arcing']
man-07   answer_contains ['fan filter']  REQUIRED ['fan filter']
mix-01   answer_contains ['CT-FN-02', '6']  REQUIRED ['CT-FN-02', '\\b6\\b|\\bsix\\b']
mix-02   answer_contains ['UX-BT-02', '0']  REQUIRED ['UX-BT-02', '\\b(?:0|zero|no)\\s+(?:units?\\b[^.]{0,30}?)?(?:in stock|available|stock)\\b|\\bstock(?: level| count| is)?\\s*[:=]?\\s*(?:0|zero|none)\\b|not in stock|out of stock|not available|unavailable', 'Zwolle']


In [3]:
# The no-stock pattern accepts the three no-stock answers and rejects the two in-stock ones.
says_none = ["CT-SR-03 is not in stock.", "There are 0 units of CT-SR-03 in stock.", "Stock: 0 at every depot."]
says_four = ["CT-SR-03 has 4 in stock at Houten (firmware 5.0 devices).", "CT-SR-03 has 4 in stock and 0 open orders."]
assert all(re.search(NO_STOCK, text, re.IGNORECASE) for text in says_none)
assert not any(re.search(NO_STOCK, text, re.IGNORECASE) for text in says_four)
print("check :", "PASS" if len(SELECTED) == 6 else "FAIL")
assert len(SELECTED) == 6

check : PASS


## Step 2 : Final-answer scores

Complete `score_answer` so it checks every required fact and every expected
tool on a desk result. `REQUIRED` from step 1 holds one regular expression per
fact.

In [4]:
def run_desk(case):
    # A run that loops until the recursion limit is a failed trial, not a crash.
    try:
        return desk.invoke({"messages": [{"role": "user", "content": case["question"]}]}, LIMIT)
    except GraphRecursionError:
        return {"messages": [HumanMessage(case["question"]),
                             AIMessage("Stopped at the recursion limit.")]}


def score_answer(case, result):
    answer = str(result["messages"][-1].content)
    used = [m.name for m in result["messages"] if m.type == "tool"]
    facts_ok = all(re.search(pattern, answer, re.IGNORECASE) for pattern in REQUIRED[case["id"]])
    tools_ok = set(case["expected_tools"]) <= set(used)
    return {"id": case["id"], "facts_ok": facts_ok, "tools_ok": tools_ok,
            "used": used, "answer": answer}


desk_results = []
for case in SELECTED:
    result = run_desk(case)
    row = score_answer(case, result)
    row["messages"] = result["messages"]
    desk_results.append(row)
    print(f"{row['id']:8} facts {row['facts_ok']!s:5} tools {row['tools_ok']!s:5} {row['used']}")
passed = sum(r["facts_ok"] and r["tools_ok"] for r in desk_results)
print(f"{passed}/{len(desk_results)} passed")

db-01    facts True  tools True  ['run_sql', 'run_sql', 'run_sql', 'run_sql']


db-03    facts True  tools True  ['run_sql', 'run_sql', 'run_sql', 'run_sql', 'run_sql', 'run_sql', 'run_sql', 'run_sql', 'run_sql']


man-01   facts True  tools True  ['search_manuals']


man-07   facts True  tools True  ['search_manuals']


mix-01   facts True  tools True  ['search_manuals', 'run_sql', 'run_sql', 'run_sql', 'run_sql']


mix-02   facts True  tools True  ['run_sql', 'search_manuals', 'run_sql', 'run_sql', 'run_sql', 'run_sql', 'run_sql', 'run_sql']
6/6 passed


In [5]:
print("check :", "PASS" if passed >= 3 else "FAIL", f"{passed}/6")
assert passed >= 3

check : PASS 6/6


## Step 3 : Trajectory match

Compare the live man-01 trajectory with a reference. Strict compares the whole
message sequence : roles, message count and the calls in each message. The full
reference therefore holds the question, the call, a tool result and an answer ;
the short one stops after the call. Use `tool_args_match_mode="ignore"` so the
reference needs no arguments. Then score three fixed fixtures : two calls
reordered inside one message, the same calls split over two messages in reverse
order, and one call missing.

In [6]:
from agentevals.trajectory.match import create_trajectory_match_evaluator

# man-01 is a single-tool case ; db-01 often over-calls and never matches unordered.
live = next(r for r in desk_results if r["id"] == "man-01")
reference_calls = [
    {"name": name, "args": {}, "id": f"ref-{i}"}
    for i, name in enumerate(next(c["expected_tools"] for c in SELECTED if c["id"] == "man-01"))
]
short_reference = [HumanMessage(next(c["question"] for c in SELECTED if c["id"] == "man-01")),
                   AIMessage("", tool_calls=reference_calls)]
reference = short_reference + [
    ToolMessage("manual extract", tool_call_id=call["id"], name=call["name"]) for call in reference_calls
] + [AIMessage("reference answer")]

print("live roles :", [m.type for m in live["messages"]])
print("live scores (man-01)")
live_scores, short_scores = {}, {}
for mode in ("strict", "unordered"):
    evaluator = create_trajectory_match_evaluator(
        trajectory_match_mode=mode, tool_args_match_mode="ignore")
    live_scores[mode] = evaluator(outputs=live["messages"], reference_outputs=reference)["score"]
    short_scores[mode] = evaluator(outputs=live["messages"], reference_outputs=short_reference)["score"]
    print(mode, ": full reference", live_scores[mode], "| short reference", short_scores[mode])

call_a = {"name": "run_sql", "args": {"query": "SELECT 1"}, "id": "a"}
call_b = {"name": "search_manuals", "args": {"query": "fan"}, "id": "b"}
both = [AIMessage("", tool_calls=[call_a, call_b])]
reversed_calls = [AIMessage("", tool_calls=[call_b, call_a])]
one = [AIMessage("", tool_calls=[call_a])]
split = [AIMessage("", tool_calls=[call_a]), AIMessage("", tool_calls=[call_b])]
split_reversed = [AIMessage("", tool_calls=[call_b]), AIMessage("", tool_calls=[call_a])]

print("fixtures")
fixture_scores = {}
for mode in ("strict", "unordered"):
    evaluator = create_trajectory_match_evaluator(
        trajectory_match_mode=mode, tool_args_match_mode="ignore")
    fixture_scores[mode] = {
        "reordered": evaluator(outputs=reversed_calls, reference_outputs=both)["score"],
        "one_call": evaluator(outputs=one, reference_outputs=both)["score"],
        "split_reversed": evaluator(outputs=split_reversed, reference_outputs=split)["score"],
    }
    print(mode, fixture_scores[mode])

live roles : ['human', 'ai', 'tool', 'ai']
live scores (man-01)
strict : full reference True | short reference False
unordered : full reference True | short reference True
fixtures
strict {'reordered': True, 'one_call': False, 'split_reversed': False}
unordered {'reordered': True, 'one_call': False, 'split_reversed': True}


In [7]:
# Strict counts messages : the short reference has two, the live run at least four.
# Installed strict match permits reordering inside one assistant message.
assert short_scores["strict"] is False
assert live_scores["unordered"] is True
assert fixture_scores["strict"]["reordered"] is True
assert fixture_scores["unordered"]["reordered"] is True
assert fixture_scores["strict"]["one_call"] is False
assert fixture_scores["unordered"]["one_call"] is False
# Split over two messages, the order of the messages counts for strict only.
assert fixture_scores["strict"]["split_reversed"] is False
assert fixture_scores["unordered"]["split_reversed"] is True
print("check : PASS live", live_scores, "short", short_scores, "fixtures", fixture_scores)

check : PASS live {'strict': True, 'unordered': True} short {'strict': False, 'unordered': True} fixtures {'strict': {'reordered': True, 'one_call': False, 'split_reversed': False}, 'unordered': {'reordered': True, 'one_call': False, 'split_reversed': True}}


## Step 4 : One score per agent

Score the parts agent on a database case, the manuals agent on a manuals case,
and both plus the desk on a mixed case. `tools_ok` is a policy check : it asks
whether the agent used the expected tools. A specialist can still answer
correctly outside its lane, so read `facts_ok` and the answer before you call a
row wrong.

In [8]:
def agent_score(agent, case):
    result = agent.invoke({"messages": [{"role": "user", "content": case["question"]}]}, LIMIT)
    row = score_answer(case, result)
    row["agent"] = agent.name
    return row


db_case = next(c for c in SELECTED if c["id"] == "db-01")
man_case = next(c for c in SELECTED if c["id"] == "man-01")
mix_case = next(c for c in SELECTED if c["id"] == "mix-01")

per_agent = [
    agent_score(parts_agent, db_case),
    agent_score(manuals_agent, man_case),
    agent_score(parts_agent, mix_case),
    agent_score(manuals_agent, mix_case),
]
desk_mix = next(r for r in desk_results if r["id"] == "mix-01")
per_agent.append({"agent": "desk", **{k: desk_mix[k] for k in ("id", "facts_ok", "tools_ok", "used", "answer")}})
for row in per_agent:
    print(f"{row['agent']:8} {row['id']:8} facts {row['facts_ok']!s:5} tools {row['tools_ok']!s:5}")

parts    db-01    facts True  tools True 
manuals  man-01   facts True  tools True 
parts    mix-01   facts True  tools False
manuals  mix-01   facts False tools False
desk     mix-01   facts True  tools True 


In [9]:
assert any(r["agent"] == "parts" for r in per_agent)
assert any(r["agent"] == "manuals" for r in per_agent)
assert any(r["agent"] == "desk" for r in per_agent)
print("check : PASS per-agent rows")

check : PASS per-agent rows


## Step 5 : Three runs

One run is one sample of a non-deterministic system. Run the desk twice more on
the six cases and keep step 2 as the first run. pass@3 counts a case when at
least one of its three runs passes ; pass^3 counts it only when all three pass
(Anthropic, "Demystifying evals for AI agents", January 2026). Complete the two
counts.

In [10]:
K = 3
runs = {r["id"]: [bool(r["facts_ok"] and r["tools_ok"])] for r in desk_results}
for _ in range(K - 1):
    for case in SELECTED:
        row = score_answer(case, run_desk(case))
        runs[case["id"]].append(bool(row["facts_ok"] and row["tools_ok"]))
for case_id, scores in runs.items():
    print(f"{case_id:8} {scores}")
pass_at_k = sum(any(scores) for scores in runs.values())
pass_hat_k = sum(all(scores) for scores in runs.values())
print(f"pass@{K} : {pass_at_k}/{len(runs)}   pass^{K} : {pass_hat_k}/{len(runs)}")

db-01    [True, True, True]
db-03    [True, True, True]
man-01   [True, True, True]
man-07   [True, True, True]
mix-01   [True, True, True]
mix-02   [True, True, True]
pass@3 : 6/6   pass^3 : 6/6


In [11]:
assert all(len(scores) == K for scores in runs.values())
assert pass_hat_k <= pass_at_k
print("check : PASS", {"pass@3": pass_at_k, "pass^3": pass_hat_k,
                      "spread": {k: sum(v) for k, v in runs.items()}})

check : PASS {'pass@3': 6, 'pass^3': 6, 'spread': {'db-01': 3, 'db-03': 3, 'man-01': 3, 'man-07': 3, 'mix-01': 3, 'mix-02': 3}}


## If time permits

- Re-score the live man-01 trajectory without `tool_args_match_mode="ignore"` ;
  empty reference args never match a live model's tool call, so the score fails
  for the wrong reason
- Add `subset` and `superset` to the trajectory table
- Replace the fact check with a trajectory judge on one mixed case